# Notebook 4: 2016 Out-of-Sample Evaluation (MSPE)

Evaluates the Stage 2 QLoRA model on 2016 data.

For each day in 2016:
1. Look back 10 days for market deltas + events
2. Predict next 10 days of log-returns
3. Recover raw prices: `p_predicted = p_last * exp(cumsum(predicted_log_returns))`
4. Compute MSPE: `mean((predicted - actual)² / actual²)`

- **Input**: `stage2_best.pt`, `combined_commodity_data.csv`, `gdelt_events_with_market_impact.csv`
- **Output**: MSPE metrics and plots

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes

In [ ]:
import os
import math
from bisect import bisect_left, bisect_right
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_token = user_secrets.get_secret("HF_TOKEN")
login(token=HF_token)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Config

In [ ]:
@dataclass
class Config:
    # Data paths (Kaggle)
    data_dir: str = "/kaggle/input/finance-world-model-dataset"
    stage2_weights: str = "/kaggle/input/stage2-weights/stage2_best.pt"

    # Architecture
    n_features: int = 46
    d_model: int = 256
    lookback: int = 10
    horizon: int = 10
    n_transformer_layers: int = 2
    n_heads: int = 4
    ff_dim: int = 1024
    dropout: float = 0.1
    max_events_per_window: int = 8

    # Qwen
    qwen_model_id: str = "Qwen/Qwen3-4B"
    qwen_hidden: int = 2560
    lora_r: int = 16
    lora_alpha: int = 32
    lora_target_modules: tuple = ("q_proj", "v_proj")

    # Critical features for loss
    critical_features: tuple = (
        "YF_Price_Crude Oil (WTI)",
        "YF_Price_Crude Oil (Brent)",
        "YF_Price_Gold",
        "YF_Price_Natural Gas",
        "YF_Price_Silver",
        "YF_Price_Copper",
    )

    # Features using simple difference instead of log return
    diff_features: tuple = ("FRED_DFF", "FRED_DGS10", "FRED_T10YIE")

    # Evaluation period
    eval_start: str = "2016-01-01"
    eval_end: str = "2016-12-31"


cfg = Config()
print(f"Evaluating: {cfg.eval_start} to {cfg.eval_end}")

## Load & Preprocess Data

In [ ]:
# Load market data
market_df = pd.read_csv(
    f"{cfg.data_dir}/combined_commodity_data.csv",
    parse_dates=["date"],
    index_col="date",
)
feature_cols = market_df.columns.tolist()
assert len(feature_cols) == cfg.n_features, f"Expected {cfg.n_features}, got {len(feature_cols)}"

critical_indices = [feature_cols.index(f) for f in cfg.critical_features]
print(f"Features: {len(feature_cols)}")

# Compute log returns
diff_cols = [c for c in feature_cols if c in cfg.diff_features]
log_cols = [c for c in feature_cols if c not in cfg.diff_features]

deltas_df = pd.DataFrame(index=market_df.index[1:], columns=feature_cols, dtype=np.float32)

for col in log_cols:
    vals = market_df[col].values.astype(np.float64)
    vals[vals == 0] = np.nan
    vals = pd.Series(vals).ffill().bfill().values
    deltas_df[col] = np.diff(np.log(vals)).astype(np.float32)

for col in diff_cols:
    deltas_df[col] = np.diff(market_df[col].values).astype(np.float32)

deltas_df = deltas_df.clip(-0.5, 0.5).fillna(0.0)
print(f"Log returns shape: {deltas_df.shape}")

In [ ]:
# Load events and build date-sorted index
events_df = pd.read_csv(f"{cfg.data_dir}/gdelt_events_with_market_impact.csv")
events_df["event_time"] = pd.to_datetime(events_df["event_time"])
events_df = events_df[events_df["market_impact"] == 1].reset_index(drop=True)
print(f"Market-impact events: {len(events_df)}")

event_dates_sorted = events_df["event_time"].sort_values().values
event_idx_by_date = events_df.sort_values("event_time").index.values
event_timestamps = [pd.Timestamp(d) for d in event_dates_sorted]


def find_events_in_window(start_date, end_date):
    left = bisect_left(event_timestamps, pd.Timestamp(start_date))
    right = bisect_right(event_timestamps, pd.Timestamp(end_date))
    return event_idx_by_date[left:right].tolist()

In [ ]:
def format_event_text(event_row, market_df, feature_cols):
    """Build event text with ALL market features prepended as context."""
    event_date = event_row["event_time"]
    summary = event_row["event_summary"]

    available = market_df.index[market_df.index <= event_date]
    if len(available) == 0:
        available = market_df.index[:1]
    market_date = available[-1]
    market_row = market_df.loc[market_date]

    parts = []
    for col in feature_cols:
        val = market_row[col]
        if pd.isna(val):
            continue
        short_name = col.replace("YF_Price_", "").replace("YF_Volume_", "Vol_").replace("FRED_", "")
        if "Price" in col or "SP500" in col:
            parts.append(f"{short_name} ${val:.2f}")
        elif "DGS" in col or "DFF" in col or "T10YIE" in col:
            parts.append(f"{short_name} {val:.2f}%")
        elif "Volume" in col:
            parts.append(f"{short_name} {val:.0f}")
        else:
            parts.append(f"{short_name} {val:.2f}")
    context = ", ".join(parts)

    date_str = event_date.strftime("%Y-%m-%d")
    return f"Market on {date_str}: {context}\nEvent ({date_str}): {summary}"


event_formatted_texts = []
for _, row in events_df.iterrows():
    event_formatted_texts.append(format_event_text(row, market_df, feature_cols))

print(f"Formatted {len(event_formatted_texts)} event texts")

## Dataset

In [ ]:
class CommodityEventDataset(Dataset):
    def __init__(self, deltas_df, raw_df, events_df, event_texts, cfg,
                 start_date, end_date):
        self.cfg = cfg
        self.event_texts = event_texts

        mask = (deltas_df.index >= start_date) & (deltas_df.index <= end_date)
        self.deltas = deltas_df[mask].values.astype(np.float32)
        self.dates = deltas_df[mask].index

        raw_mask = (raw_df.index >= start_date) & (raw_df.index <= end_date)
        self.raw_values = raw_df[raw_mask].values.astype(np.float32)
        self.raw_dates = raw_df[raw_mask].index

        self.events_df = events_df

        total_needed = cfg.lookback + cfg.horizon
        self.n_samples = max(0, len(self.deltas) - total_needed + 1)
        print(f"Dataset [{start_date} to {end_date}]: {self.n_samples} samples")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        L = self.cfg.lookback
        H = self.cfg.horizon

        market_deltas = torch.from_numpy(self.deltas[idx : idx + L])
        target_deltas = torch.from_numpy(self.deltas[idx + L : idx + L + H])

        anchor_date = self.dates[idx + L - 1]
        raw_idx = self.raw_dates.get_loc(anchor_date)
        anchor_state = torch.from_numpy(self.raw_values[raw_idx])

        window_start = self.dates[idx]
        window_end = self.dates[idx + L - 1]
        event_indices = find_events_in_window(window_start, window_end)
        event_indices = event_indices[: self.cfg.max_events_per_window]
        n_events = len(event_indices)

        texts = [self.event_texts[i] for i in event_indices]

        max_k = self.cfg.max_events_per_window
        event_mask = torch.ones(max_k, dtype=torch.bool)
        event_mask[:n_events] = False

        return {
            "market_deltas": market_deltas,
            "anchor_state": anchor_state,
            "target_deltas": target_deltas,
            "event_texts": texts,
            "event_mask": event_mask,
        }


def collate_fn(batch):
    return {
        "market_deltas": torch.stack([s["market_deltas"] for s in batch]),
        "anchor_state": torch.stack([s["anchor_state"] for s in batch]),
        "target_deltas": torch.stack([s["target_deltas"] for s in batch]),
        "event_texts": [s["event_texts"] for s in batch],
        "event_mask": torch.stack([s["event_mask"] for s in batch]),
    }

In [ ]:
eval_ds = CommodityEventDataset(
    deltas_df, market_df, events_df, event_formatted_texts, cfg,
    cfg.eval_start, cfg.eval_end,
)

eval_loader = DataLoader(
    eval_ds, batch_size=1, shuffle=False,
    collate_fn=collate_fn, num_workers=0,
)

print(f"Eval samples: {len(eval_ds)}")

## Model Architecture (Stage 2)

In [ ]:
class MarketMLP(nn.Module):
    def __init__(self, n_features, d_model):
        super().__init__()
        self.linear = nn.Linear(n_features, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.linear(x))


class FiLMLayer(nn.Module):
    def __init__(self, conditioning_dim, feature_dim):
        super().__init__()
        self.gamma_linear = nn.Linear(conditioning_dim, feature_dim)
        self.beta_linear = nn.Linear(conditioning_dim, feature_dim)
        nn.init.ones_(self.gamma_linear.bias)
        nn.init.zeros_(self.gamma_linear.weight)
        nn.init.zeros_(self.beta_linear.bias)
        nn.init.zeros_(self.beta_linear.weight)

    def forward(self, x, conditioning):
        gamma = self.gamma_linear(conditioning).unsqueeze(1)
        beta = self.beta_linear(conditioning).unsqueeze(1)
        return gamma * x + beta


class EventMarketCrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True,
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, market_seq, event_reprs, event_mask):
        attn_out, _ = self.cross_attn(
            query=market_seq, key=event_reprs, value=event_reprs,
            key_padding_mask=event_mask,
        )
        return self.norm(market_seq + attn_out)


class TemporalTransformer(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, ff_dim, dropout, seq_len):
        super().__init__()
        self.pos_encoding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x):
        x = x + self.pos_encoding
        return self.encoder(x)


class PredictionHead(nn.Module):
    def __init__(self, d_model, n_features, critical_indices):
        super().__init__()
        self.head = nn.Linear(d_model, n_features)
        self.register_buffer("critical_indices", torch.tensor(critical_indices, dtype=torch.long))

    def forward(self, h, target_deltas=None):
        predicted = self.head(h)
        loss = None
        if target_deltas is not None:
            pred_critical = predicted[:, :, self.critical_indices]
            target_critical = target_deltas[:, :, self.critical_indices]
            loss = F.mse_loss(pred_critical, target_critical)
        return predicted, loss

In [ ]:
class LiveEventEncoder(nn.Module):
    """Event encoder with 4-bit quantized Qwen3-4B + LoRA (QLoRA)."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

        self.tokenizer = AutoTokenizer.from_pretrained(cfg.qwen_model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

        qwen = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )

        lora_config = LoraConfig(
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            target_modules=list(cfg.lora_target_modules),
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
        self.qwen = get_peft_model(qwen, lora_config)

        self.projection = nn.Linear(cfg.qwen_hidden, cfg.d_model)
        self.no_event_embed = nn.Parameter(torch.zeros(cfg.d_model))

    def encode_texts(self, texts, device):
        if len(texts) == 0:
            return torch.zeros(0, self.cfg.d_model, device=device)

        inputs = self.tokenizer(
            texts, return_tensors="pt", padding=True,
            truncation=True, max_length=512,
        ).to(device)

        with torch.no_grad():
            outputs = self.qwen(**inputs, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1]
        attention_mask = inputs["attention_mask"]
        seq_lens = attention_mask.sum(dim=1) - 1
        embeds = last_hidden[torch.arange(len(texts), device=device), seq_lens]
        return self.projection(embeds.float())

    def forward(self, event_texts_batch, event_mask, device):
        B = len(event_texts_batch)
        max_k = self.cfg.max_events_per_window
        d = self.cfg.d_model

        all_texts = []
        text_to_idx = {}
        mappings = []

        for b, texts in enumerate(event_texts_batch):
            for i, t in enumerate(texts):
                if t not in text_to_idx:
                    text_to_idx[t] = len(all_texts)
                    all_texts.append(t)
                mappings.append((b, i, text_to_idx[t]))

        if all_texts:
            all_embeds = self.encode_texts(all_texts, device)
        else:
            all_embeds = torch.zeros(0, d, device=device)

        e_reprs = torch.zeros(B, max_k, d, device=device)
        for b, i, tidx in mappings:
            e_reprs[b, i] = all_embeds[tidx]

        valid_mask = ~event_mask
        n_valid = valid_mask.sum(dim=1, keepdim=True).clamp(min=1)
        masked = e_reprs * valid_mask.unsqueeze(-1).float()
        e_global = masked.sum(dim=1) / n_valid.float()

        no_events = valid_mask.sum(dim=1) == 0
        if no_events.any():
            e_global[no_events] = self.no_event_embed.unsqueeze(0)

        return e_global, e_reprs


class FiLMWorldModelStage2(nn.Module):
    """Stage 2 model with live QLoRA Qwen event encoding."""

    def __init__(self, cfg, critical_indices):
        super().__init__()
        d = cfg.d_model
        self.market_mlp = MarketMLP(cfg.n_features, d)
        self.film_anchor = FiLMLayer(cfg.n_features, d)
        self.live_event_encoder = LiveEventEncoder(cfg)
        self.film_event = FiLMLayer(d, d)
        self.cross_attn = EventMarketCrossAttention(d, cfg.n_heads, cfg.dropout)
        self.temporal_transformer = TemporalTransformer(
            d, cfg.n_heads, cfg.n_transformer_layers,
            cfg.ff_dim, cfg.dropout, cfg.lookback,
        )
        self.prediction_head = PredictionHead(d, cfg.n_features, critical_indices)

    def forward(self, market_deltas, anchor_state, event_texts_batch, event_mask,
                target_deltas=None):
        h = self.market_mlp(market_deltas)
        h = self.film_anchor(h, anchor_state)
        e_global, e_reprs = self.live_event_encoder(
            event_texts_batch, event_mask, market_deltas.device,
        )
        h = self.film_event(h, e_global)
        h = self.cross_attn(h, e_reprs, event_mask)
        h = self.temporal_transformer(h)
        predicted, loss = self.prediction_head(h, target_deltas)
        return predicted, loss

## Load Model & Evaluate

In [ ]:
model_s2 = FiLMWorldModelStage2(cfg, critical_indices)
model_s2.load_state_dict(torch.load(cfg.stage2_weights, weights_only=True))
model_s2.eval()
print("Stage 2 model loaded for evaluation.")

In [ ]:
all_mspe = []

with torch.no_grad():
    for i, batch in enumerate(eval_loader):
        market_deltas = batch["market_deltas"].to(DEVICE)
        anchor_state = batch["anchor_state"].to(DEVICE)
        event_texts = batch["event_texts"]
        event_mask = batch["event_mask"].to(DEVICE)
        target_deltas = batch["target_deltas"]

        predicted_deltas, _ = model_s2(
            market_deltas, anchor_state, event_texts, event_mask,
        )
        predicted_deltas = predicted_deltas.cpu()

        anchor_prices = batch["anchor_state"]

        pred_cum_log = predicted_deltas.cumsum(dim=1)
        actual_cum_log = target_deltas.cumsum(dim=1)

        pred_prices = anchor_prices.unsqueeze(1) * torch.exp(pred_cum_log)
        actual_prices = anchor_prices.unsqueeze(1) * torch.exp(actual_cum_log)

        safe_actual = actual_prices.clamp(min=1e-8)
        pct_error_sq = ((pred_prices - actual_prices) / safe_actual) ** 2
        mspe_per_feature = pct_error_sq.mean(dim=1).squeeze(0)
        all_mspe.append(mspe_per_feature)

        if (i + 1) % 50 == 0:
            print(f"Evaluated {i + 1}/{len(eval_ds)} samples")

print(f"\nEvaluation complete: {len(all_mspe)} samples")

In [ ]:
# Aggregate MSPE results
all_mspe_tensor = torch.stack(all_mspe)
mean_mspe = all_mspe_tensor.mean(dim=0)

# Overall MSPE
overall_mspe = mean_mspe.mean().item()
print(f"Overall MSPE (all features): {overall_mspe:.6f} ({overall_mspe*100:.4f}%)")

# Critical features MSPE
critical_mspe = mean_mspe[critical_indices].mean().item()
print(f"Critical features MSPE: {critical_mspe:.6f} ({critical_mspe*100:.4f}%)")

# Per-feature MSPE table
print(f"\n{'Feature':<40} {'MSPE':>12} {'RMSPE %':>10}")
print("-" * 62)
for i, col in enumerate(feature_cols):
    mspe_val = mean_mspe[i].item()
    rmspe = np.sqrt(mspe_val) * 100
    marker = " *" if i in critical_indices else ""
    print(f"{col:<40} {mspe_val:>12.6f} {rmspe:>9.2f}%{marker}")

print(f"\n* = critical feature")

In [ ]:
# Plot MSPE per critical feature over time
all_mspe_np = all_mspe_tensor.numpy()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax_idx, (feat_name, feat_idx) in enumerate(zip(cfg.critical_features, critical_indices)):
    if ax_idx >= len(axes):
        break
    mspe_series = all_mspe_np[:, feat_idx]
    rmspe_series = np.sqrt(mspe_series) * 100
    axes[ax_idx].plot(rmspe_series, alpha=0.6, linewidth=0.8)
    axes[ax_idx].set_title(feat_name.replace('YF_Price_', ''), fontsize=10)
    axes[ax_idx].set_ylabel("RMSPE %")
    axes[ax_idx].set_xlabel("Sample")
    mean_val = rmspe_series.mean()
    axes[ax_idx].axhline(y=mean_val, color="r", linestyle="--", alpha=0.5,
                         label=f"Mean: {mean_val:.2f}%")
    axes[ax_idx].legend(fontsize=8)

plt.suptitle("2016 Out-of-Sample RMSPE (%) - Critical Features", fontsize=12)
plt.tight_layout()
plt.show()